In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import geopandas as gpd
import xgboost as xgb
import shap

print("pandas:", pd.__version__)
print("xgboost:", xgb.__version__)
print("shap:", shap.__version__)
print("All libraries loaded successfully!")


pandas: 2.3.3
xgboost: 3.2.0
shap: 0.49.1
All libraries loaded successfully!


In [7]:
import pandas as pd
import requests
import os

# Create all folders
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

print("Folders ready!")


Folders ready!


In [8]:
url = "https://data.cdc.gov/api/views/swc5-untb/rows.csv?accessType=DOWNLOAD"

print("Downloading CDC PLACES... (may take 30-60 seconds)")
response = requests.get(url, timeout=120)

with open("data/raw/cdc_places.csv", "wb") as f:
    f.write(response.content)

print("Done! File size:", round(len(response.content) / 1024 / 1024, 1), "MB")

Done! File size: 50.9 MB


In [9]:
cdc = pd.read_csv("data/raw/cdc_places.csv")
print("Total rows:", len(cdc))
print("\nAll available measures:")
for m in cdc['Measure'].unique():
    print(" -", m)

Total rows: 229298

All available measures:
 - Arthritis among adults
 - Current asthma among adults
 - Stroke among adults
 - Any disability among adults
 - Obesity among adults
 - Diagnosed diabetes among adults
 - Depression among adults
 - Hearing disability among adults
 - Binge drinking among adults
 - High blood pressure among adults
 - Vision disability among adults
 - Mobility disability among adults
 - Self-care disability among adults
 - Frequent mental distress among adults
 - High cholesterol among adults who have ever been screened
 - Frequent physical distress among adults
 - Colorectal cancer screening among adults aged 45–75 years
 - No leisure-time physical activity among adults
 - Chronic obstructive pulmonary disease among adults
 - Independent living disability among adults
 - Coronary heart disease among adults
 - Mammography use among women aged 50-74 years
 - Visits to doctor for routine checkup within the past year among adults
 - Taking medicine to control hig

/tmp/ipykernel_46272/2783326364.py:1: DtypeWarning: Columns (10,11) have mixed types. Specify dtype option on import or set low_memory=False.
  cdc = pd.read_csv("data/raw/cdc_places.csv")


In [12]:
print("Columns in your CDC file:")
for col in cdc.columns:
    print(f"  '{col}'")

Columns in your CDC file:
  'Year'
  'StateAbbr'
  'StateDesc'
  'LocationName'
  'DataSource'
  'Category'
  'Measure'
  'Data_Value_Unit'
  'Data_Value_Type'
  'Data_Value'
  'Data_Value_Footnote_Symbol'
  'Data_Value_Footnote'
  'Low_Confidence_Limit'
  'High_Confidence_Limit'
  'TotalPopulation'
  'TotalPop18plus'
  'LocationID'
  'CategoryID'
  'MeasureId'
  'DataValueTypeID'
  'Short_Question_Text'
  'Geolocation'


In [13]:
# First let's see what measures are available
print("Available measures:")
for m in cdc['Measure'].unique():
    print(f"  '{m}'")

Available measures:
  'Arthritis among adults'
  'Current asthma among adults'
  'Stroke among adults'
  'Any disability among adults'
  'Obesity among adults'
  'Diagnosed diabetes among adults'
  'Depression among adults'
  'Hearing disability among adults'
  'Binge drinking among adults'
  'High blood pressure among adults'
  'Vision disability among adults'
  'Mobility disability among adults'
  'Self-care disability among adults'
  'Frequent mental distress among adults'
  'High cholesterol among adults who have ever been screened'
  'Frequent physical distress among adults'
  'Colorectal cancer screening among adults aged 45–75 years'
  'No leisure-time physical activity among adults'
  'Chronic obstructive pulmonary disease among adults'
  'Independent living disability among adults'
  'Coronary heart disease among adults'
  'Mammography use among women aged 50-74 years'
  'Visits to doctor for routine checkup within the past year among adults'
  'Taking medicine to control high

In [14]:
# These are the exact measure names in YOUR file
mental_health_measures = [
    'Frequent mental distress among adults',
    'Depression among adults'
]

# Your file doesn't have GeographicLevel
# LocationID with 5 digits = county level
cdc['LocationID'] = cdc['LocationID'].astype(str).str.zfill(5)

# Filter: only 5-digit LocationIDs are counties (state IDs are 2 digits)
cdc_filtered = cdc[
    (cdc['LocationID'].str.len() == 5) &
    (cdc['Measure'].isin(mental_health_measures))
][['LocationName', 'StateDesc', 'LocationID', 'Measure', 'Data_Value', 'TotalPopulation']]

cdc_filtered.columns = ['county', 'state', 'fips', 'measure', 'value', 'population']

# Drop rows with no value
cdc_filtered = cdc_filtered.dropna(subset=['value'])

cdc_filtered.to_csv("data/raw/cdc_filtered.csv", index=False)
print(f"Saved! Shape: {cdc_filtered.shape}")
print(cdc_filtered.head(6))

Saved! Shape: (11828, 6)
      county     state   fips                  measure  value  population
15   Flagler   Florida  12035  Depression among adults   18.3      131439
18  Seminole   Florida  12117  Depression among adults   16.4      484271
31   Wheeler   Georgia  13309  Depression among adults   19.0        7081
39   Douglas  Illinois  17041  Depression among adults   22.6       19629
52   Fremont      Iowa  19071  Depression among adults   20.1        6458
86   Webster  Nebraska  31181  Depression among adults   17.6        3351


In [18]:
CENSUS_API_KEY = "895d6234502008be149c7c42967e35bba0c41252"

variables = "B19013_001E,B01002_001E,B17001_002E,B28002_004E,B01003_001E"

census_url = (
    f"https://api.census.gov/data/2021/acs/acs5"
    f"?get=NAME,{variables}"
    f"&for=county:*"
    f"&key={CENSUS_API_KEY}"
)

print("Downloading Census ACS data...")
response = requests.get(census_url, timeout=120)
data = response.json()

census_df = pd.DataFrame(data[1:], columns=data[0])
census_df.columns = ['name', 'median_income', 'median_age',
                      'poverty_pop', 'broadband_pop', 'total_pop',
                      'state_code', 'county_code']

# Combine state + county into FIPS code
census_df['fips'] = census_df['state_code'] + census_df['county_code']

# Convert to numbers
for col in ['median_income', 'median_age', 'poverty_pop', 'broadband_pop', 'total_pop']:
    census_df[col] = pd.to_numeric(census_df[col], errors='coerce')

census_df.to_csv("data/raw/census_acs.csv", index=False)
print(f"Saved! Shape: {census_df.shape}")
print(census_df.head(3))

Saved! Shape: (3221, 9)
                      name  median_income  median_age  poverty_pop  \
0  Autauga County, Alabama          62660        38.5         7847   
1  Baldwin County, Alabama          64346        43.4        20598   
2  Barbour County, Alabama          36422        40.2         5890   

   broadband_pop  total_pop state_code county_code   fips  
0          18679      58239         01         001  01001  
1          76602     227131         01         003  01003  
2           5872      25259         01         005  01005  


In [21]:
# We already have Census data - let's use it to create a proxy SCI
# Using broadband access + population density as social connectivity proxy

census_df = pd.read_csv("data/raw/census_acs.csv")

# Create proxy Social Connectivity Score
# Higher broadband + younger population = higher digital social connectivity
census_df['fips'] = census_df['fips'].astype(str).str.zfill(5)
census_df['broadband_pop'] = pd.to_numeric(census_df['broadband_pop'], errors='coerce')
census_df['total_pop'] = pd.to_numeric(census_df['total_pop'], errors='coerce')
census_df['median_age'] = pd.to_numeric(census_df['median_age'], errors='coerce')

# Broadband rate as % of population
census_df['broadband_rate'] = census_df['broadband_pop'] / census_df['total_pop']

# Proxy SCI = normalized broadband rate inverted by age
# (younger + more broadband = more digitally connected)
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()

census_df['sci_proxy'] = (
    scaler.fit_transform(census_df[['broadband_rate']]) * 0.6 +
    scaler.fit_transform(1 / census_df[['median_age']]) * 0.4
)

sci_proxy = census_df[['fips', 'sci_proxy']].copy()
sci_proxy.columns = ['fips', 'sci']
sci_proxy.to_csv("data/raw/sci_within_county.csv", index=False)
print(f"Proxy SCI saved! Shape: {sci_proxy.shape}")
print(sci_proxy.head(5))

Proxy SCI saved! Shape: (3221, 2)
    fips       sci
0  01001  0.465245
1  01003  0.449673
2  01005  0.324659
3  01007  0.333543
4  01009  0.403525


In [22]:
files = {
    "CDC PLACES":           "data/raw/cdc_filtered.csv",
    "Social Capital (SCI)": "data/raw/sci_within_county.csv",
    "Census ACS":           "data/raw/census_acs.csv"
}

print("Verifying all downloads...\n")
all_good = True

for name, path in files.items():
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"  {name}: {df.shape[0]:,} rows x {df.shape[1]} cols")
    else:
        print(f"  MISSING: {name} — re-run that cell")
        all_good = False

if all_good:
    print("\nAll 3 datasets ready!")
    print("Step 2 complete — ready for Step 3!")
else:
    print("\nSome files missing — check errors above.")

Verifying all downloads...

  CDC PLACES: 11,828 rows x 6 cols
  Social Capital (SCI): 3,221 rows x 2 cols
  Census ACS: 3,221 rows x 9 cols

All 3 datasets ready!
Step 2 complete — ready for Step 3!
